# Кластеризация и снижение размерности

**Датасет:** Wine Recognition Dataset — химический состав вин (3 сорта винограда).



In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
    homogeneity_completeness_v_measure,
)

sns.set_theme(style='whitegrid')

## Загрузка и EDA

In [ ]:
wine = load_wine()
feature_names = wine.feature_names
df = pd.DataFrame(wine.data, columns=feature_names)
true_labels = wine.target

print(df.shape)
print(df.isnull().sum().sum())
df.describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(), cmap='coolwarm', center=0, ax=ax)
ax.set_title('Корреляция признаков')
plt.tight_layout()
plt.show()

## D1: подмножество признаков (без целевого)

In [ ]:
D1 = StandardScaler().fit_transform(df.values)
print('D1:', D1.shape)

## D2: PCA (2 компоненты)

In [ ]:
pca = PCA(n_components=2, random_state=42)
D2 = pca.fit_transform(D1)
print('D2:', D2.shape)
print('Доля объяснённой дисперсии:', pca.explained_variance_ratio_)
print('Суммарно:', pca.explained_variance_ratio_.sum().round(3))

## D3: t-SNE (2 компоненты)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
D3 = tsne.fit_transform(D1)
print('D3:', D3.shape)

## Визуализация D2 и D3

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(D2[:, 0], D2[:, 1], c=true_labels, cmap='viridis', alpha=0.8, edgecolors='white')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('D2: PCA')

axes[1].scatter(D3[:, 0], D3[:, 1], c=true_labels, cmap='viridis', alpha=0.8, edgecolors='white')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].set_title('D3: t-SNE')

plt.tight_layout()
plt.show()

In [ ]:
inertia = []
K_range = range(1, 11)
for k in K_range:
    inertia.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(D1).inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, np.sqrt(inertia), 'o-')
ax.set_xlabel('Число кластеров k')
ax.set_ylabel('sqrt(inertia)')
ax.set_title('Метод локтя (D1)')
plt.show()

## Кластеризация

In [ ]:
N_CLUSTERS = 3

DBSCAN_PARAMS = {
    'D1': {'eps': 2.2, 'min_samples': 3},
    'D2': {'eps': 0.6, 'min_samples': 5},
    'D3': {'eps': 2.0, 'min_samples': 5},
}

def get_clusterers(ds_name):
    return {
        'KMeans': KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10),
        'Agglomerative': AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward'),
        'DBSCAN': DBSCAN(**DBSCAN_PARAMS[ds_name]),
    }

def cluster_dataset(X, name):
    results = {}
    for method, model in get_clusterers(name).items():
        labels = model.fit_predict(X)
        results[method] = labels
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        print(f'{name} | {method}: кластеров={n_clusters}, шум={np.sum(labels == -1)}')
    return results

datasets = {'D1': D1, 'D2': D2, 'D3': D3}
all_labels = {d: cluster_dataset(X, d) for d, X in datasets.items()}

In [ ]:
def compute_metrics(X, y_true, labels):
    mask = labels != -1
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        sil = np.nan
    else:
        sil = silhouette_score(X[mask], labels[mask])

    h, c, v = homogeneity_completeness_v_measure(y_true, labels)
    return {
        'Silhouette': sil,
        'ARI': adjusted_rand_score(y_true, labels),
        'AMI': adjusted_mutual_info_score(y_true, labels),
        'Homogeneity': h,
        'Completeness': c,
        'V-measure': v,
    }

rows = []
for ds_name, methods in all_labels.items():
    X_data = datasets[ds_name]
    for method, labels in methods.items():
        row = {'Датасет': ds_name, 'Метод': method, **compute_metrics(X_data, true_labels, labels)}
        rows.append(row)

metrics_df = pd.DataFrame(rows).set_index(['Датасет', 'Метод'])
metrics_df.round(3)

## Визуализация кластеров

In [ ]:
def plot_clusters_2d(X2d, labels, title, ax):
    scatter = ax.scatter(X2d[:, 0], X2d[:, 1], c=labels, cmap='tab10', alpha=0.8, edgecolors='white')
    ax.set_title(title)
    return scatter

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
for i, ds_name in enumerate(['D1', 'D2', 'D3']):
    X2d = D2 if ds_name == 'D1' else datasets[ds_name]
    for j, method in enumerate(['KMeans', 'Agglomerative', 'DBSCAN']):
        plot_clusters_2d(X2d, all_labels[ds_name][method], f'{ds_name}: {method}', axes[i, j])
plt.tight_layout()
plt.show()

In [ ]:
plot_df = metrics_df.reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=plot_df, x='Метод', y='Silhouette', hue='Датасет', ax=axes[0])
axes[0].set_title('Silhouette (больше — лучше)')

sns.barplot(data=plot_df, x='Метод', y='ARI', hue='Датасет', ax=axes[1])
axes[1].set_title('ARI (больше — лучше)')

plt.tight_layout()
plt.show()

## Выводы

**D2 vs D3:** на D3 (t-SNE) кластеры визуально разделены явнее, чем на D2 (PCA), т.к. t-SNE сохраняет локальные расстояния между объектами.

**Лучший метод по датасетам** — см. таблицу метрик выше (Silhouette без эталона, ARI/V-measure с эталоном `class`).

In [ ]:
best = metrics_df.reset_index().sort_values('Silhouette', ascending=False)
print('Топ по Silhouette:')
print(best.head(3)[['Датасет', 'Метод', 'Silhouette', 'ARI', 'V-measure']])

print('\nЛучший метод для каждого датасета (по Silhouette):')
print(metrics_df.reset_index().sort_values('Silhouette', ascending=False).groupby('Датасет').first()[['Метод', 'Silhouette', 'ARI']])